# Multi-Feature Delivery Time Prediction Model

This notebook demonstrates how to build a PyTorch neural network to predict delivery times based on distance, time of day, and traffic conditions.

In [ ]:
## Magic cmd to automatically reload modules when they have been changed.
%load_ext autoreload
%autoreload 2

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from plot_support import plot_numpyarr
from plot_support import plot_regression_line, dynamic_plot

# Ensure reproducibility by setting a manual seed
torch.manual_seed(42)

# Set print options for better readability of tensors
torch.set_printoptions(sci_mode=False)

In [ ]:
# --- 1. Load and Inspect Data ---

# Read the dataset containing delivery information
# Note: Update the path if running locally
df_in = pd.read_csv(r'/Users/xiaoyao/Library/CloudStorage/GoogleDrive-xiaowen19920501@gmail.com/My Drive/PyTorch_Practice/data_with_features.csv')

# Visualize the raw relationship between distance and delivery time
# This helps confirm that distance is a primary factor, though not the only one
plot_numpyarr(df_in['distance_miles'].values, df_in['delivery_time_minutes'].values)

# Display the first few rows to understand the structure
df_in.head(5)

In [ ]:
# --- 2. Feature Engineering ---

# We need to capture traffic patterns. 
# Logic: Rush hour is defined as 8am-10am OR 4pm-7pm (16-19), 
# but ONLY on weekdays (is_weekend == 0).

# Using np.where is faster than applying a function row-by-row
# Syntax: np.where(condition, value_if_true, value_if_false)
df_in['is_rush_hour'] = np.where(
    (df_in['is_weekend'] == 0) & 
    ((df_in['time_of_day_hours'] >= 8) & (df_in['time_of_day_hours'] <= 10) | 
     (df_in['time_of_day_hours'] >= 16) & (df_in['time_of_day_hours'] <= 19)),
    1, 0
)

# --- 3. Normalization (Z-Score) ---

# Neural networks struggle if inputs have vastly different scales (e.g., 0-1 vs 0-100).
# We standardize continuous features to have Mean = 0 and Std Dev = 1.
df_in['distance_miles_norm'] = (df_in['distance_miles'] - df_in['distance_miles'].mean()) / df_in['distance_miles'].std()
df_in['time_of_day_hours_norm'] = (df_in['time_of_day_hours'] - df_in['time_of_day_hours'].mean()) / df_in['time_of_day_hours'].std()

# Create a cleaner view of the input features and target
new_df_in = df_in[['distance_miles_norm', 'time_of_day_hours_norm','is_weekend', 'is_rush_hour', 'delivery_time_minutes']].head(10)
new_df_in.head(5)

In [ ]:
# --- 4. Prepare PyTorch Tensors ---

# Convert Pandas DataFrames to PyTorch Tensors.
# float32 is the standard data type for deep learning computations.

# Input Features (X): Normalized Distance, Normalized Time, Weekend Flag, Rush Hour Flag
torch_data_in = torch.tensor(
    new_df_in[['distance_miles_norm', 'time_of_day_hours_norm', 'is_weekend', 'is_rush_hour']].values, 
    dtype=torch.float32
)

# Target Variable (y): Delivery Time
# .view(-1, 1) reshapes the 1D array into a 2D column vector (N rows, 1 column)
# This ensures shape compatibility with the model's output.
torch_data_out = torch.tensor(
    new_df_in['delivery_time_minutes'].values, 
    dtype=torch.float32
).view(-1, 1)

print(f'Input Shape: {torch_data_in.shape}')
print(f'Output Shape: {torch_data_out.shape}')
print("Sample Input Data:\n", torch_data_in)

In [ ]:
# --- 5. Define Model Architecture ---

def init_model():
    """
    Initializes the neural network, optimizer, and loss function.
    """
    # Set seed again inside function for safety if called multiple times
    torch.manual_seed(42)

    # Sequential model container
    model = nn.Sequential(
        # Layer 1: 4 Inputs -> 64 Hidden Neurons
        nn.Linear(4, 64),
        # Activation: ReLU introduces non-linearity
        nn.ReLU(),
        
        # Layer 2: 64 Hidden -> 32 Hidden Neurons
        nn.Linear(64, 32),
        nn.ReLU(),
        
        # Output Layer: 32 Hidden -> 1 Output (Prediction)
        nn.Linear(32, 1)
    )

    # Optimizer: Stochastic Gradient Descent (SGD) with learning rate 0.01
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    
    # Loss Function: Mean Squared Error (Standard for regression)
    loss_fn = nn.MSELoss()  

    return model, optimizer, loss_fn

# Initialize model components
model, optimizer, loss_fn = init_model()

In [ ]:
# --- 6. Training Loop ---

def train_model(model, optimizer, loss_fn, data_in, data_out, epochs=100):
    """
    Executes the training process.
    """
    losses = []

    for epoch in range(epochs):
        # 1. Forward Pass: Compute predictions
        output = model(data_in)

        # 2. Compute Loss: Compare predictions to actuals
        loss = loss_fn(output, data_out)

        # 3. Zero Gradients: Clear previous gradients to prevent accumulation
        optimizer.zero_grad()

        # 4. Backward Pass: Calculate gradients (backpropagation)
        loss.backward()

        # 5. Optimizer Step: Update weights
        optimizer.step()

        # Logging
        if (epoch+1) % 10 == 0:
            losses.append(loss.item())
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
    
    return model, losses

# Start training
# We pass the tensors created earlier
test_model, loss_history = train_model(model, optimizer, loss_fn, torch_data_in, torch_data_out, epochs=100)